# Deliverable 3 — Complete Workflow: from Images to Captions

This notebook runs the 3 trained models one after the other (no training):

```txt
 image(s) ──> 1. CLASSIFICATION (D1) ──> not a photo: set aside with its class
                      │ photo
                      ▼
             2. DENOISING (D2)
                      │
                      ▼
             3. CAPTIONING (D3: ConvNeXt + GRU with attention) ──> caption + attention maps
```

| Step | Model file | Input |
|---|---|---|
| 1. photo or not | `results/Hybrid_MobileNetV2.keras` | 224 x 224, pixels 0-255 |
| 2. denoising | `results/denoise_UNet_deep.keras` | 256 x 256, pixels 0-1 |
| 3a. image encoder | ConvNeXt-Base ImageNet, grid of 7 x 7 regions | 224 x 224 |
| 3b. caption | `results/caption_ShowAttendTell_ConvNeXtBase.keras` + `..._vocab.json` | regions + previous words |

The captioning model is the best one of `03_captioning.ipynb` (section 7, COCO val2017, beam search 5):

| model | BLEU-4 | CIDEr |
|---|---|---|
| v1 Hvass (InceptionV3 + 3 GRU) | 0.272 | 0.86 |
| v3 first attention (InceptionV3) | 0.264 | 0.84 |
| **v4 Show, Attend and Tell (ConvNeXt-Base)** | **0.339** | **1.12** |

**Quick test on one image**: put `test.jpg` at the root of the project, run sections 1 to 4. The result
(class, denoised image, caption, attention maps) is shown and saved in `results/test_result.png`.

**Demo day**: run sections 1 to 3, then change `INPUT_DIR` (and `ANNOTATIONS` if captions are given) in the last
section and run it.


In [1]:
import tensorflow as tf, keras
print("TensorFlow", tf.__version__, "| Keras", keras.__version__)

I0000 00:00:1792662106.951129   46686 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792662107.131592   46686 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792662109.790392   46686 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow 2.21.0 | Keras 3.15.1


## 1. Configuration

In [ ]:
print("[Block 1] Configuration : starting...")

import hashlib, json, os, re, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import imageio.v3 as iio
from PIL import Image   # fallback for the formats TensorFlow cannot read (TIFF, ...)
import tensorflow as tf
import keras
from keras import layers, ops
from keras.applications import ConvNeXtBase, InceptionV3, VGG16, convnext, inception_v3, vgg16
from nltk.translate.bleu_score import corpus_bleu
from pycocoevalcap.cider.cider import Cider

warnings.filterwarnings("ignore", module="nltk")

ROOT = Path("")
RAW = ROOT / "data_raw"
DATA = ROOT / "data"
RESULTS = ROOT / "results"

TEST_IMAGE = ROOT / "test.jpg"   # one image to test (section 4)
INPUT_DIR = RAW / "Dataset"      # folder to process (section 6)
ANNOTATIONS = None            # optional: COCO .json or .csv (file_name, caption)

DENOISE = True                # False = skip the denoising step
DECODING = "beam"             # "beam" or "greedy"
BEAM = 5                      # same beam width as in 03_captioning.ipynb
BATCH = 32
# Captioning model trained in 03_captioning.ipynb (the best one of its section 7):
#   "ShowAttendTell_ConvNeXtBase" (v4, best), "Attention_GRU_InceptionV3" (v3), "Hvass_GRU_InceptionV3" (v1)
# Attention maps are shown only for the models with attention (v3, v4)
CAPTION_MODEL = "ShowAttendTell_ConvNeXtBase"
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".webp", ".tif", ".tiff"}
MIN_SIZE = 16                 # smaller images (in pixels) are ignored
CONVERTED = DATA / "converted"   # images converted to PNG (formats that TensorFlow cannot read)

# Same classes and order as deliverable 1
CLASS_NAMES = ["Painting", "Photo", "Schematics", "Sketch", "Text"]
PHOTO = CLASS_NAMES.index("Photo")

print("[Block 1] OK")

## 2. Loading the Models

The models are read from `results/`. If a file is missing, run the notebook that trains it.

In [3]:
print("[Block 2] Loading the models : starting...")

# Custom layer of the v4 captioning models: same code as 03_captioning.ipynb (section 5), needed to load them
@keras.saving.register_keras_serializable(package="Leyenda")
class AttentionGRU(layers.Layer):
    """GRU that looks at the image regions before writing each word (Show, Attend and Tell).

    Inputs: [words (batch, T, E), regions (batch, R, D), initial state (batch, units)]
    Outputs: [GRU states (batch, T, units), contexts (batch, T, D), attention weights (batch, T, R)]
    """
    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.supports_masking = True   # padding steps are skipped (state unchanged)
        self.att_regions = layers.Dense(units, use_bias=False, name="att_regions")   # region -> attention space
        self.att_state = layers.Dense(units, name="att_state")                       # GRU memory -> attention space
        self.att_score = layers.Dense(1, use_bias=False, name="att_score")           # one score per region
        self.gate = layers.Dense(1, activation="sigmoid", name="gate")               # how much image to use (0 to 1)
        self.cell = layers.GRUCell(units, name="cell")                               # one GRU step

    def build(self, input_shape):
        words, regions, _ = input_shape
        self.att_regions.build(regions)
        self.att_state.build((None, self.units))
        self.att_score.build((None, None, self.units))
        self.gate.build((None, self.units))
        self.cell.build((None, words[-1] + regions[-1]))   # input of the GRU = word + context

    def call(self, inputs, mask=None, training=False):
        words, regions, h0 = inputs
        keys = self.att_regions(regions)   # computed once per image
        m = mask[0] if isinstance(mask, (list, tuple)) else mask
        m = ops.ones(ops.shape(words)[:2]) if m is None else ops.cast(m, words.dtype)
        u = self.units
        f = ops.shape(regions)[-1]

        def step(previous, x):
            # previous = [h, context, alpha] side by side (tf.scan needs one tensor)
            h, (word, real) = previous[:, :u], x
            # 1. Attention: a score for each region, from the GRU memory h (what was written so far)
            scores = self.att_score(ops.tanh(keys + self.att_state(h)[:, None, :]))[..., 0]
            alpha = ops.softmax(scores, axis=-1)                                # R weights that sum to 1
            context = self.gate(h) * ops.sum(alpha[..., None] * regions, axis=1)   # weighted mean of the regions
            # 2. GRU step: reads the word AND the context
            new_h = self.cell(ops.concatenate([word, context], axis=-1), [h], training=training)[1][0]
            h = real[:, None] * new_h + (1 - real[:, None]) * h               # padding: keep the old state
            return ops.concatenate([h, context, alpha], axis=-1)

        start = ops.concatenate([h0, ops.zeros_like(regions[:, 0]), ops.zeros_like(regions[:, :, 0])], axis=-1)
        out = tf.scan(step, (ops.transpose(words, (1, 0, 2)), ops.transpose(m, (1, 0))), initializer=start)
        out = ops.transpose(out, (1, 0, 2))   # (batch, T, units + D + R)
        return out[..., :u], out[..., u:u + f], out[..., u + f:]

    def compute_mask(self, inputs, mask=None):
        m = mask[0] if isinstance(mask, (list, tuple)) else mask
        return [m, m, m]

    def compute_output_shape(self, input_shape):
        words, regions, _ = input_shape
        return ((words[0], words[1], self.units), (words[0], words[1], regions[-1]),
                (words[0], words[1], regions[1]))

    def get_config(self):
        return {**super().get_config(), "units": self.units}

start = time.time()
classifier = keras.models.load_model(RESULTS / "Hybrid_MobileNetV2.keras", compile=False)
denoiser = keras.models.load_model(RESULTS / "denoise_UNet_deep.keras", compile=False)
with open(RESULTS / f"caption_{CAPTION_MODEL}_vocab.json") as f:
    caption_config = json.load(f)   # vocabulary + name of the CNN used in training
decoder = keras.models.load_model(RESULTS / caption_config["model"], compile=False)

def build_encoder(name):
    # Same CNN as in 03_captioning.ipynb (section 3)
    if name == "InceptionV3":
        cnn = InceptionV3(include_top=True, weights="imagenet")
        return keras.Model(cnn.input, cnn.get_layer("avg_pool").output, name="InceptionV3_avg_pool"), inception_v3.preprocess_input
    if name == "InceptionV3_grid":
        # 8 x 8 regions -> 64 vectors of 2048 numbers (attention model)
        cnn = InceptionV3(include_top=True, weights="imagenet")
        block = cnn.get_layer("mixed10").output
        grid = keras.layers.Reshape((-1, block.shape[-1]), name="regions")(block)
        return keras.Model(cnn.input, grid, name="InceptionV3_mixed10"), inception_v3.preprocess_input
    if name == "ConvNeXtBase_grid":
        # Last block before the classifier: 7 x 7 regions -> 49 vectors of 1024 numbers
        cnn = ConvNeXtBase(include_top=True, weights="imagenet")
        block = cnn.get_layer("convnext_base_head_gap").input                  # (7, 7, 1024)
        block = cnn.get_layer("convnext_base_head_layernorm")(block)           # same normalisation as the classifier
        grid = keras.layers.Reshape((-1, block.shape[-1]), name="regions")(block)
        return keras.Model(cnn.input, grid, name="ConvNeXtBase_grid"), convnext.preprocess_input
    if name == "VGG16":
        cnn = VGG16(include_top=True, weights="imagenet")
        return keras.Model(cnn.input, cnn.get_layer("fc2").output, name="VGG16_fc2"), vgg16.preprocess_input
    raise ValueError(f"unknown encoder {name}")

encoder, preprocess_input = build_encoder(caption_config["encoder"])

VOCAB = caption_config["vocab"]
MAX_WORDS = caption_config["max_words"]
TOKEN_START = VOCAB.index(caption_config["mark_start"])
TOKEN_END = VOCAB.index(caption_config["mark_end"])
CLF_SIZE = tuple(classifier.input_shape[1:3])     # 224 x 224
DENOISE_SIZE = tuple(denoiser.input_shape[1:3])   # 256 x 256
ENC_SIZE = tuple(encoder.input_shape[1:3])        # 224 x 224 (ConvNeXt), 299 x 299 (InceptionV3)
assert decoder.output_shape[-1] == len(VOCAB), "the vocabulary file does not match the captioning model"

for name, m in (("classifier", classifier), ("denoiser", denoiser), ("encoder", encoder), ("decoder", decoder)):
    print(f"  {name:10s} {m.name:28s} {m.count_params():>12,} weights | input {m.input_shape}")
print(f"  vocabulary: {len(VOCAB)} words, captions of at most {MAX_WORDS} words | loaded in {time.time() - start:.0f} s")

print("[Block 2] OK")

[Block 2] Loading the models : starting...


W0000 00:00:1792662113.590557   46686 gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was false.
I0000 00:00:1792662113.591144   46686 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13466 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


  classifier Hybrid_MobileNetV2              3,013,957 weights | input (None, 224, 224, 3)
  denoiser   UNet_deep                       1,951,939 weights | input (None, 256, 256, 3)
  encoder    ConvNeXtBase_grid              87,566,464 weights | input (None, 224, 224, 3)
  decoder    ShowAttendTell_ConvNeXtBase    11,630,353 weights | input [(None, None), (None, 49, 1024)]
  vocabulary: 10000 words, captions of at most 19 words | loaded in 7 s
[Block 2] OK


## 3. The Workflow Steps

Each function uses the same preprocessing as in the notebook where the model was trained.

In [ ]:
print("[Block 3] Workflow steps : definition...")

AUTOTUNE = tf.data.AUTOTUNE

def read_rgb(path):
    # RGB, 8 bits (a 16-bit PNG is converted, a grey or RGBA image gets 3 channels)
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, dtype=tf.uint8, expand_animations=False)
    img.set_shape([None, None, 3])
    return tf.cast(img, tf.float32)

def classify(paths):
    # Step 1: probabilities of the 5 classes
    ds = tf.data.Dataset.from_tensor_slices(paths).map(lambda p: tf.image.resize(read_rgb(p), CLF_SIZE), AUTOTUNE)
    return classifier.predict(ds.batch(BATCH).prefetch(AUTOTUNE), verbose=0)

def load_01(paths):
    # Photos in 256 x 256, pixels in [0, 1]
    ds = tf.data.Dataset.from_tensor_slices(paths).map(
        lambda p: tf.clip_by_value(tf.round(tf.image.resize(read_rgb(p), DENOISE_SIZE, antialias=True)), 0, 255) / 255.0,
        AUTOTUNE)   # same as deliverable 2
    return np.concatenate([b.numpy() for b in ds.batch(BATCH)])

def denoise(images):
    # Step 2: denoising
    return np.concatenate([denoiser(images[s:s + BATCH], training=False).numpy() for s in range(0, len(images), BATCH)])

def encode(images):
    # Step 3a: CNN vectors
    x = tf.image.resize(np.asarray(images, dtype="float32") * 255.0, ENC_SIZE)
    return encoder.predict(preprocess_input(x), batch_size=BATCH, verbose=0)

# Step 3b: captions (same code as 03_captioning.ipynb, section 5.1)
def tokens_to_text(tokens):
    words = []
    for t in tokens[1:]:
        if t == TOKEN_END or t == 0:
            break
        words.append(VOCAB[t])
    return " ".join(words)

def next_word_probs(feats, tokens):
    out = decoder({"transfer_values_input": feats, "decoder_input": tokens}, training=False)
    return np.array(out[:, -1], dtype="float64")

def greedy_captions(features, max_words=MAX_WORDS):
    feats = np.asarray(features, dtype="float32")
    tokens = np.zeros((len(feats), max_words + 2), dtype="int32"); tokens[:, 0] = TOKEN_START
    done = np.zeros(len(feats), dtype=bool)
    for t in range(max_words + 1):
        probs = next_word_probs(feats, tokens[:, :t + 1]); probs[:, :2] = 0
        word = probs.argmax(axis=1); word[done] = 0
        tokens[:, t + 1] = word
        done |= word == TOKEN_END
        if done.all():
            break
    return [tokens_to_text(row) for row in tokens]

def beam_captions(features, beam=BEAM, max_words=MAX_WORDS, alpha=0.7):
    V = len(VOCAB)
    feats = np.repeat(np.asarray(features, dtype="float32"), beam, axis=0)
    B = len(feats) // beam
    tokens = np.zeros((B * beam, max_words + 2), dtype="int32"); tokens[:, 0] = TOKEN_START
    score = np.full((B, beam), -1e9); score[:, 0] = 0.0
    length = np.zeros((B, beam))
    done = np.zeros((B, beam), dtype=bool)
    for t in range(max_words + 1):
        logp = np.log(np.maximum(next_word_probs(feats, tokens[:, :t + 1]), 1e-12)).reshape(B, beam, V)
        logp[..., :2] = -1e9
        cand = np.where(done[..., None], -1e9, score[..., None] + logp)
        cand[..., 0] = np.where(done, score, -1e9)
        flat = cand.reshape(B, beam * V)
        best = np.argsort(-flat, axis=1)[:, :beam]
        src, word = best // V, best % V
        tokens = tokens[(np.arange(B)[:, None] * beam + src).ravel()]
        tokens[:, t + 1] = word.ravel()
        score = np.take_along_axis(flat, best, axis=1)
        was_done = np.take_along_axis(done, src, axis=1)
        length = np.take_along_axis(length, src, axis=1) + ~was_done
        done = was_done | (word == TOKEN_END)
        if done.all():
            break
    pick = (score / np.maximum(length, 1) ** alpha).argmax(axis=1)
    return [tokens_to_text(tokens[b * beam + pick[b]]) for b in range(B)]

def write_captions(features):
    fn = beam_captions if DECODING == "beam" else greedy_captions
    return sum((fn(features[s:s + 200]) for s in range(0, len(features), 200)), [])

# Attention maps (only for the attention models): where the network looks for each word
HAS_ATTENTION = any(layer.name == "decoder_attention" for layer in decoder.layers)

def attention_probe(model):
    # Function (inputs) -> (attention weights, gates) of each word. The gate (v4 only) is the share of the image
    # used to write the word: 0 = the word is written from the sentence only, 1 = from the image only
    layer = model.get_layer("decoder_attention")
    if not isinstance(layer, AttentionGRU):   # v3: no gate
        probe = keras.Model(model.inputs, layer.output[-1])
        return lambda x: (np.asarray(probe(x, training=False)), None)
    probe = keras.Model(model.inputs, [layer.output[0], layer.output[-1], model.get_layer("decoder_transfer_map").output])
    def run(x):
        states, weights, h0 = [np.asarray(t) for t in probe(x, training=False)]
        previous = np.concatenate([h0[:, None], states[:, :-1]], axis=1)   # memory used to choose the regions of word t
        return weights, np.asarray(layer.gate(previous))[..., 0]           # same gate as inside AttentionGRU
    return run

if HAS_ATTENTION:
    attention = attention_probe(decoder)

def show_attention(image, caption, weights, gates=None, title="", cols=6):
    # Same as 03_captioning.ipynb (section 7.3): one small image per word, red = region used
    grid = int(np.sqrt(weights.shape[1]))   # 8 for InceptionV3, 7 for ConvNeXt
    h, w = image.shape[:2]
    words = caption.split()
    gates = [None] * len(words) if gates is None else gates
    rows = int(np.ceil((len(words) + 1) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 2.9 * rows), squeeze=False)
    axes = axes.ravel()
    axes[0].imshow(image); axes[0].set_title(title or "image", fontsize=11)
    for ax, word, wt, g in zip(axes[1:], words, weights, gates):
        ax.imshow(image)
        # The CNN saw the image resized to a square: the grid covers the whole image
        ax.imshow(wt.reshape(grid, grid), cmap="jet", alpha=0.5, interpolation="bilinear", extent=(0, w, h, 0))
        info = f"max {wt.max():.2f}" + ("" if g is None else f" | image {g:.0%}")
        ax.set_title(f"{word}\n{info}", fontsize=11)
    for ax in axes:
        ax.axis("off")
    plt.suptitle(caption, fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

def show_attention_maps(table, photos, n=3):
    # Attention maps of the first n captioned photos (on the denoised image, the one the CNN saw)
    if not HAS_ATTENTION:
        print(f"  no attention maps: {CAPTION_MODEL} has no attention layer")
        return
    captions = table.set_index("path")["caption"]
    for j in range(min(n, len(photos["paths"]))):
        path = photos["paths"][j]
        words = captions[path].split()
        tokens = np.array([[TOKEN_START] + [VOCAB.index(w) for w in words]], dtype="int32")
        weights, gates = attention({"transfer_values_input": np.asarray(photos["features"][j:j + 1], dtype="float32"),
                                    "decoder_input": tokens})
        show_attention(np.clip(photos["denoised"][j], 0, 1), captions[path], weights[0, :len(words)],
                       None if gates is None else gates[0, :len(words)], title=Path(path).name)

# Input files: everything that is not a usable image is ignored, with the reason
def prepare_image(path):
    # One file -> (path readable by TensorFlow, None), or (None, reason why it is ignored)
    path = Path(path)
    if not path.is_file():
        return None, "file not found"
    if not os.access(path, os.R_OK):
        return None, "no read permission"
    if path.stat().st_size == 0:
        return None, "empty file (0 bytes)"
    if path.suffix.lower() not in IMAGE_EXTENSIONS:
        return None, f"not an image extension ({path.suffix or 'none'})"
    try:   # 1. format read by TensorFlow (JPEG, PNG, BMP, GIF, WebP)
        shape = read_rgb(path.as_posix()).shape
        if min(shape[:2]) < MIN_SIZE:
            return None, f"too small ({shape[1]} x {shape[0]} pixels)"
        return path.as_posix(), None
    except Exception:
        pass
    try:   # 2. other formats (TIFF, ...): converted to PNG with Pillow
        with Image.open(path) as im:
            im = im.convert("RGB")
            if min(im.size) < MIN_SIZE:
                return None, f"too small ({im.size[0]} x {im.size[1]} pixels)"
            CONVERTED.mkdir(parents=True, exist_ok=True)
            out = CONVERTED / f"{path.stem}_{hashlib.md5(path.as_posix().encode()).hexdigest()[:8]}.png"
            im.save(out)
        return out.as_posix(), None
    except Exception as e:
        return None, f"cannot be read as an image ({type(e).__name__})"

def find_files(folder):
    # All the files of a folder and its sub-folders (hidden files excluded)
    folder = Path(folder)
    if not folder.is_dir():
        print(f"  {folder}: folder not found")
        return []
    return sorted(p.as_posix() for p in folder.rglob("*") if p.is_file() and not p.name.startswith("."))

def prepare_images(files):
    # Files -> (original paths, readable paths) of the usable images + table of the ignored files
    originals, readable, ignored = [], [], []
    for f in files:
        path, reason = prepare_image(f)
        if path is None:
            ignored.append({"file_name": Path(f).name, "reason": reason})
        else:
            originals.append(f); readable.append(path)
    return originals, readable, pd.DataFrame(ignored, columns=["file_name", "reason"])

def list_images(folder):
    # Usable images of a folder (readable paths)
    return prepare_images(find_files(folder))[1]

def run_pipeline(paths, denoise_photos=DENOISE, names=None):
    # Full workflow on a list of readable images (see prepare_images): one row per image
    paths = list(paths)
    if not paths:   # nothing to do (empty folder, or only ignored files)
        table = pd.DataFrame(columns=["path", "file_name", "class", "p_photo", "caption"])
        empty = np.zeros((0,) + DENOISE_SIZE + (3,), "float32")
        return table, {"original": empty, "denoised": empty, "paths": [], "features": None}, \
            pd.DataFrame(columns=["seconds", "ms per image"])
    timing = {}
    t0 = time.time()
    probs = classify(paths)
    timing["1. classification"] = time.time() - t0
    table = pd.DataFrame({"path": paths, "file_name": names or [Path(p).name for p in paths],
                          "class": [CLASS_NAMES[i] for i in probs.argmax(axis=1)],
                          "p_photo": probs[:, PHOTO].round(4), "caption": ""})
    is_photo = (probs.argmax(axis=1) == PHOTO)
    photo_paths = table.loc[is_photo, "path"].tolist()
    photos = load_01(photo_paths) if photo_paths else np.zeros((0,) + DENOISE_SIZE + (3,), "float32")
    if denoise_photos and len(photos):
        t0 = time.time()
        cleaned = denoise(photos)
        timing["2. denoising"] = time.time() - t0
    else:
        cleaned = photos
    if len(photos):
        t0 = time.time()
        feats = encode(cleaned)
        timing["3a. CNN encoder"] = time.time() - t0
        t0 = time.time()
        table.loc[is_photo, "caption"] = write_captions(feats)
        timing["3b. caption writing"] = time.time() - t0
    else:
        feats = None
    timing = pd.Series(timing, name="seconds").to_frame()
    timing["ms per image"] = timing["seconds"] / len(paths) * 1000
    return table, {"original": photos, "denoised": cleaned, "paths": photo_paths, "features": feats}, timing

print("[Block 3] OK")

## 4. Quick Test on One Image

Put an image (noisy or not) at the root of the project with the name `test.jpg` (or change `TEST_IMAGE` in
section 1), then run this cell. It shows each step of the workflow on this image:

1. **classification**: probability of each of the 5 classes. If the image is not a photo, it is set aside, as in
   the real workflow (`FORCE_CAPTION = True` writes a caption anyway, to see what the network would say);
2. **denoising**: the image before and after the denoiser;
3. **caption** written by the network, then the **attention maps**: for each word, where the network looks
   (red), *max* = how small the area is, *image %* = share of the image used for this word (see
   `03_captioning.ipynb`, sections 7.3 and 7.4).

The summary figure is saved in `results/test_result.png`.


In [ ]:
print("[Block 4] Quick test on one image : starting...")

FORCE_CAPTION = False   # True = write a caption even if the image is not classified as a photo

def caption_one_image(path, force=FORCE_CAPTION):
    path = Path(path)
    readable, reason = prepare_image(path)
    if readable is None:
        print(f"  {path}: {reason}. Put an image with this name at the root of the project (or change TEST_IMAGE)")
        return None
    start = time.time()
    # 1. Classification
    probs = classify([readable])[0]
    label = CLASS_NAMES[probs.argmax()]
    is_photo = probs.argmax() == PHOTO
    print(f"  1. class: {label} (p = {probs.max():.2f}) -> " + ("photo: captioned" if is_photo else
          "not a photo: set aside" + (", captioned anyway (FORCE_CAPTION)" if force else "")))
    # 2. Denoising, 3. caption
    original = load_01([readable])
    cleaned = denoise(original) if DENOISE else original
    caption, features = "", None
    if is_photo or force:
        features = encode(cleaned)
        caption = write_captions(features)[0]
    print(f"  3. caption: {caption or '(none)'} | {time.time() - start:.1f} s")

    # Summary figure: probabilities, input, denoised + caption
    fig, axes = plt.subplots(1, 3, figsize=(17, 5), gridspec_kw={"width_ratios": [1, 1.3, 1.3]})
    colors = ["#c0392b" if i == probs.argmax() else "#9e9e9e" for i in range(len(CLASS_NAMES))]
    axes[0].barh(CLASS_NAMES, probs, color=colors); axes[0].set_xlim(0, 1)
    axes[0].set_title("1. Classification (probabilities)", fontweight="bold")
    axes[1].imshow(read_rgb(readable).numpy().astype("uint8")); axes[1].set_title(f"input: {path.name}", fontweight="bold")
    axes[2].imshow(np.clip(cleaned[0], 0, 1))
    axes[2].set_title("2. denoised\n3. " + (f'"{caption}"' if caption else "no caption (not a photo)"),
                      fontweight="bold")
    for ax in axes[1:]:
        ax.axis("off")
    plt.tight_layout()
    out = RESULTS / f"{path.stem}_result.png"
    plt.savefig(out, dpi=100, bbox_inches="tight")
    plt.show()
    print(f"  figure saved in {out}")

    # Attention maps (on the denoised image, the one the CNN saw)
    if caption and HAS_ATTENTION:
        words = caption.split()
        tokens = np.array([[TOKEN_START] + [VOCAB.index(w) for w in words]], dtype="int32")
        weights, gates = attention({"transfer_values_input": np.asarray(features, dtype="float32"),
                                    "decoder_input": tokens})
        show_attention(np.clip(cleaned[0], 0, 1), caption, weights[0, :len(words)],
                       None if gates is None else gates[0, :len(words)], title="denoised")
    return {"file": path.name, "class": label, "p_photo": float(probs[PHOTO]), "caption": caption}

test_result = caption_one_image(TEST_IMAGE)

print("[Block 4] OK")


## 5. Check on Images of Known Type

- **non-photos** from the test split of deliverable 1 (not seen in training): they must be set aside;
- **photos** from COCO val2017 and from `data_raw/Dataset` (noisy scans): they must get a caption.

In [ ]:
print("[Block 5] Self-check : starting...")

from sklearn.model_selection import train_test_split

def deliverable1_test_split():
    # Test images of deliverable 1 (same cleaning and split as 01_photo_ou_pas.ipynb)
    index = pd.read_csv(DATA / "index.csv")
    index = index[index["category"].isin(CLASS_NAMES)]
    index = index[index["readable"] & (index[["width", "height"]].min(axis=1) >= 32)
                  & ~index.duplicated("md5", keep="first")]
    _, rest = train_test_split(index, test_size=0.30, stratify=index["category"], random_state=42)
    _, test = train_test_split(rest, test_size=0.50, stratify=rest["category"], random_state=42)
    return test

rng = np.random.default_rng(42)
check = []
if (DATA / "index.csv").exists():
    test_d1 = deliverable1_test_split()
    for category, part in test_d1[test_d1["category"] != "Photo"].groupby("category"):
        readable = [p for p in part["path"] if os.access(p, os.R_OK)]
        check += [(p, category) for p in rng.choice(readable, size=min(20, len(readable)), replace=False)]
coco_val = find_files(RAW / "coco" / "val2017")
check += [(p, "Photo (COCO)") for p in rng.choice(coco_val, size=min(30, len(coco_val)), replace=False)]
noisy = find_files(RAW / "Dataset")
check += [(p, "Photo (noisy scan)") for p in rng.choice(noisy, size=min(30, len(noisy)), replace=False)]

check = [(prepare_image(p)[0], t) for p, t in check]
check = [(p, t) for p, t in check if p is not None]   # unusable files are left out
check_table, _, _ = run_pipeline([p for p, _ in check])
check_table["true type"] = [t for _, t in check]
check_table["captioned"] = check_table["caption"] != ""
summary = check_table.groupby("true type").agg(images=("path", "size"), sent_to_captioning=("captioned", "mean"))
display(summary.round(3))
print("  not a photo, but captioned:", check_table[~check_table["true type"].str.startswith("Photo") & check_table.captioned]["file_name"].tolist())
print("  photo, but set aside:", check_table[check_table["true type"].str.startswith("Photo") & ~check_table.captioned][["file_name", "class"]].values.tolist())

print("[Block 5] OK")

## 6. Running the Workflow on `INPUT_DIR`

Every file of the folder (and its sub-folders) is checked first. The files that cannot be used are
**ignored** and listed with the reason (empty file, not an image, damaged file, too small...). The formats
that TensorFlow cannot read (TIFF, ...) are converted to PNG in `data/converted/`. An empty folder gives an
empty result instead of an error.

In [ ]:
print("[Block 6] Workflow : starting...")

def process_folder(folder):
    files = find_files(folder)
    originals, readable, ignored = prepare_images(files)
    print(f"  {folder}: {len(files)} files -> {len(readable)} images to process, {len(ignored)} ignored")
    if len(ignored):
        display(ignored)
    table, photos, timing = run_pipeline(readable, names=[Path(p).name for p in originals])
    out = RESULTS / f"pipeline_{Path(folder).name}.csv"
    pd.concat([table.drop(columns="path"), ignored.assign(**{"class": "ignored"})]).to_csv(out, index=False)
    if not len(table):
        print(f"  no image to process -> {out}")
        return table, photos
    print(f"  {table['caption'].ne('').sum()} photos captioned, {table['caption'].eq('').sum()} images set aside "
          f"-> {out}")
    display(table["class"].value_counts().rename("images").to_frame())
    display(timing.round(3))
    print(f"  total: {timing['ms per image'].sum():.1f} ms per image")
    return table, photos

def show_gallery(table, photos, n=8):
    # Photos (input / denoised + caption), then images set aside
    k = min(n, len(photos["paths"]))
    if k:
        fig, axes = plt.subplots(2, k, figsize=(3.6 * k, 8), squeeze=False)
        captions = table.set_index("path").loc[photos["paths"][:k], "caption"]
        for j in range(k):
            axes[0, j].imshow(photos["original"][j]); axes[0, j].axis("off"); axes[0, j].set_title("input", fontsize=9)
            axes[1, j].imshow(np.clip(photos["denoised"][j], 0, 1)); axes[1, j].axis("off")
            cap = captions.iloc[j]
            axes[1, j].set_title("\n".join(cap[i:i + 30] for i in range(0, len(cap), 30)), fontsize=9)
        plt.suptitle("Photos: input (top) -> denoised + caption (bottom)", fontweight="bold")
        plt.tight_layout(); plt.show()
    aside = table[table["caption"] == ""].head(n)
    if len(aside):
        fig, axes = plt.subplots(1, len(aside), figsize=(3 * len(aside), 3.4), squeeze=False)
        for ax, (_, row) in zip(axes[0], aside.iterrows()):
            ax.imshow(read_rgb(row["path"]).numpy().astype("uint8")); ax.axis("off")
            ax.set_title(f"{row['class']}\np(photo) = {row['p_photo']:.2f}", fontsize=9)
        plt.suptitle("Set aside (not a photo)", fontweight="bold")
        plt.tight_layout(); plt.show()

table, photos = process_folder(INPUT_DIR)
display(table[["file_name", "class", "p_photo", "caption"]].head(15))
show_gallery(table, photos)
show_attention_maps(table, photos)

print("[Block 6] OK")

## 8. Production and Maintenance

- **Environment**: `make up` starts the Docker container (TensorFlow 2.21 + GPU, Jupyter on port 8888).
- **Needed files**: the 3 `.keras` models and `caption_<model>_vocab.json` in `results/` (ConvNeXt-Base is
  downloaded by Keras the first time, 355 MB). The class `AttentionGRU` (section 2) must be defined before
  loading the v4 captioning model: it is the same code as in `03_captioning.ipynb`, keep both identical.
- **Retraining**: each model is retrained in its own notebook; keep the same file names. The vocabulary and the
  captioning model must come from the same run (checked in block 2).
- **New data**: add TouNum images with captions in COCO format and retrain `03_captioning.ipynb`.
- **Speed**: see the timing table of section 6.

## 9. Demo Day

Run this cell.

In [ ]:
print("[Block 9] Demo : starting...")

INPUT_DIR = RAW / "Demo"   # <- the folder given on demo day

demo_table, demo_photos = process_folder(INPUT_DIR)
display(demo_table[["file_name", "class", "p_photo", "caption"]])
show_gallery(demo_table, demo_photos, n=8)
show_attention_maps(demo_table, demo_photos, n=3)   # where the network looks (attention model only)

print("[Block 9] OK")

[Block 9] Demo : starting...
  data_raw/Dataset: 148 images


KeyboardInterrupt: 